# TraceGC: Deterministic Context Compaction Demo

Deterministic, receipt-preserving context compaction middleware for AI agents.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/athishio/trace-gc/blob/main/demo/colab_demo.ipynb)

> [!NOTE]
> This notebook uses a real, previously-verified model response (see [`scripts/logs/answer_quality_20260731_150352.jsonl`](https://github.com/athishio/trace-gc/blob/main/scripts/logs/answer_quality_20260731_150352.jsonl)) rather than a live API call, so it runs with zero setup. To verify live yourself, add your own `GEMINI_API_KEY` and re-run cells 4 and 5.

In [ ]:
# Install the library and Google GenAI SDK
!pip install trace-gc google-genai

In [ ]:
import os
import json
from trace_gc.events import validate_event
from trace_gc.compactor import compact_events

# Define Scenario 5 events (Multi-Stage Abandonment)
events = [
    {"id": "e001", "type": "decision", "timestamp": 1000, "parent_id": None, "content": "Starting system configuration"},
    {"id": "e002", "type": "set_var", "timestamp": 1010, "parent_id": "e001", "key": "refill_rate", "value": 5},
    {"id": "a01", "type": "decision", "timestamp": 1020, "parent_id": "e002", "content": "Attempting optimization"},
    {"id": "a02", "type": "set_var", "timestamp": 1030, "parent_id": "a01", "key": "refill_rate", "value": 8},
    {"id": "ab01", "type": "abandon", "timestamp": 1040, "parent_id": "a02", "ref_to": ["a01"]},
    {"id": "e003", "type": "decision", "timestamp": 1050, "parent_id": "e002", "content": "Resuming baseline configuration"}
]

print("=== RAW EVENTS ===")
for e in events:
    print(e)

# Run compaction locally
compaction_result = compact_events(events)

print("\n=== COMPACTED PROMPT ===")
print(compaction_result["prompt"])

### The Stale-Context Ambiguity

In the **raw event history**, the `abandon` event is invisible when rendering history as a text log. Consequently, the downstream LLM sees:
1. `refill_rate` initialized to `5`.
2. `refill_rate` updated to `8` under an "optimization attempt".
3. The system resuming.

Without knowing that the optimization branch was abandoned, the LLM naturally assumes that the final state of `refill_rate` is `8`.

In contrast, **TraceGC** parses the parent-child relationships and identifies that the branch containing `refill_rate = 8` was pruned by the `Dead-Branch Sweeper`. It replaces the pruned branch with a lightweight `[RECEIPT a01]` token, ensuring that the active context remains correct and unambiguous while retaining historical awareness.

In [ ]:
# Raw history prompt evaluation
question = "Question: What is the final refill_rate value set?"
uncompacted_prompt = """Event history:

Starting system configuration
refill_rate = 5
Attempting optimization
refill_rate = 8
Resuming baseline configuration

""" + question

print("=== UNCOMPACTED PROMPT SENT TO GEMINI ===")
print(uncompacted_prompt)

gemini_key = os.environ.get("GEMINI_API_KEY")
if gemini_key:
    # Live API call
    from google import genai
    client = genai.Client(api_key=gemini_key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=uncompacted_prompt,
        config={"temperature": 0.0}
    )
    response_text = response.text.strip()
    print(f"\nLive Gemini Response: {response_text}")
else:
    # Cached response from logs/answer_quality_20260731_150352.jsonl
    response_text = "8"
    print(f"\nCached Gemini Response: {response_text} (Incorrect, latched onto stale value)")

In [ ]:
# Compacted history prompt evaluation
compacted_prompt = """Event history:

""" + compaction_result["prompt"] + "\n\n" + question

print("=== COMPACTED PROMPT SENT TO GEMINI ===")
print(compacted_prompt)

if gemini_key:
    # Live API call
    from google import genai
    client = genai.Client(api_key=gemini_key)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=compacted_prompt,
        config={"temperature": 0.0}
    )
    response_text = response.text.strip()
    print(f"\nLive Gemini Response: {response_text}")
else:
    # Cached response from logs/answer_quality_20260731_150352.jsonl
    response_text = "5"
    print(f"\nCached Gemini Response: {response_text} (Correct, successfully resolved via TraceGC)")

### Takeaway

1. **TraceGC prevents reasoning failures**: Large language models easily get confused by superseded, overridden, or abandoned context that remains in the history.
2. **Zero-AI latency and cost**: Compaction is performed entirely locally on a deterministic state graph, without invoking any secondary LLM calls.
3. **No Information Loss**: Even though the failed attempts are pruned from the prompt prefix, they remain recoverable on-demand using `get_receipt(graph, "a01")`.